In [ ]:
from datetime import date
from pathlib import Path

import polars as pl

programs_all = pl.read_parquet(Path.cwd().resolve() / "train_programs_preprocess.parquet")

In [8]:
pl.Config(set_tbl_rows=-1)

In [ ]:
programs_all = pl.read_parquet(Path.cwd().resolve() / "train_programs_preprocess.parquet")
# 1. レース場・日付単位で集計
daily = (
    programs_all.group_by(["race_stadium_number", "race_date"])
    .agg(
        pl.col("racer_assigned_motor_top_2_percent").max().alias("m2p_max"),
        pl.col("racer_assigned_motor_top_3_percent").max().alias("m3p_max"),
    )
    .sort(["race_stadium_number", "race_date"])
)


# 2. モーター交換日を検出
changes = (
    daily.with_columns(pl.col("m2p_max").shift(1).over("race_stadium_number").alias("prev_m2p"))
    .filter((pl.col("m2p_max") == 0) & (pl.col("m3p_max") == 0) & (pl.col("prev_m2p") > 0))
    .select(
        "race_stadium_number",
        pl.col("race_date").alias("motor_valid_from"),
    )
)

# 3. 各レース場の最初の観測日を取得
initial = daily.group_by("race_stadium_number").agg(pl.col("race_date").min().alias("motor_valid_from"))

# 4. 初期世代と交換日を結合し、期間を生成
motor_generations = (
    pl.concat([initial, changes])
    .unique(subset=["race_stadium_number", "motor_valid_from"])
    .sort(["race_stadium_number", "motor_valid_from"])
    .with_columns((pl.col("motor_valid_from").shift(-1).over("race_stadium_number") - pl.duration(days=1)).alias("motor_valid_end"))
    .with_columns(pl.col("motor_valid_end").fill_null(date(2050, 12, 31)))
    .with_columns((pl.col("motor_valid_from").cum_count().over("race_stadium_number") - 1).alias("motor_generation"))
    .select(
        pl.col("race_stadium_number").alias("race_stadium_number_valid"),
        "motor_valid_from",
        "motor_valid_end",
        "motor_generation",
    )
)

motor_generations


race_stadium_number_valid,motor_valid_from,motor_valid_end,motor_generation
i64,date,date,u32
1,2025-07-15,2025-12-26,0
1,2025-12-27,2050-12-31,1
2,2025-07-15,2025-08-05,0
2,2025-08-06,2026-08-05,1
2,2026-08-06,2050-12-31,2
3,2025-07-20,2026-05-10,0
3,2026-05-11,2050-12-31,1
4,2025-07-15,2026-06-17,0
4,2026-06-18,2050-12-31,1


In [88]:
motor_generations.write_parquet(Path.cwd().resolve() / "motor_generations.parquet")
